In [0]:
-- Incremental load for mart_stock_features

merge into analytics.mart_stock_features t
using (

  with base as (

    select
      ticker,
      trade_date,
      close,
      volume,

      lag(close) over (
        partition by ticker
        order by trade_date
      ) as prev_close
    from analytics.fact_stock_daily

    where trade_date >= (
      select dateadd(day, -60, max(trade_date))
        from analytics.mart_stock_features
    )
  ),

  returns as (
      select *,
      (close - prev_close) / nullif(prev_close, 0) as daily_return
    from base 
    where prev_close is not null
  ),

  rolling as (
    select *,
      (close /lag(close, 5) over (partition by ticker order by trade_date)) - 1 as return_5d,
      (close /lag(close, 20) over (partition by ticker order by trade_date)) - 1 as return_20d,
      (close /lag(close, 50) over (partition by ticker order by trade_date)) - 1 as return_50d,

      stddev(daily_return) over (
        partition by ticker
        order by trade_date
        rows between 20 preceding and 1 preceding
      ) as vol_20d,

      avg(volume) over (
        partition by ticker
        order by trade_date
        rows between 20 preceding and 1 preceding
      ) as avg_volume_20d,

      max(close) over (
        partition by ticker
        order by trade_date
        rows between 20 preceding and 1 preceding
      ) as high_20d

    from returns
  ),

  features as (
    select *,
      (close / high_20d) as pct_of_20d_high,

      case 
        when volume > avg_volume_20d then 1 else 0
      end as is_volume_spike,

      case 
        when close >= high_20d * 0.99 
          and return_20d > 0
          and volume > avg_volume_20d
        then 1 else 0
      end as is_breakout_candidate,

      row_number() over (
        partition by trade_date
        order by return_20d desc
      ) as momentum_rank

    from rolling
  )

    select *
    from features
    where return_50d is not null
) s

  on t.ticker = s.ticker
  and t.trade_date = s.trade_date

  when matched then update set
    close = s.close,
    volume = s.volume,
    daily_return = s.daily_return,
    return_5d = s.return_5d,
    return_20d = s.return_20d,
    return_50d = s.return_50d,
    vol_20d = s.vol_20d,
    avg_volume_20d = s.avg_volume_20d,
    high_20d = s.high_20d,
    pct_of_20d_high = s.pct_of_20d_high,
    is_volume_spike = s.is_volume_spike,
    is_breakout_candidate = s.is_breakout_candidate,
    momentum_rank = s.momentum_rank
  
  when not matched then insert (
    ticker,
    trade_date,
    close,
    volume,
    prev_close,
    daily_return,
    return_5d,
    return_20d,
    return_50d,
    vol_20d,
    avg_volume_20d,
    high_20d,
    pct_of_20d_high,
    is_volume_spike,
    is_breakout_candidate,
    momentum_rank
  )
  values (
    s.ticker,
    s.trade_date,
    s.close,
    s.volume,
    s.prev_close,
    s.daily_return,
    s.return_5d,
    s.return_20d,
    s.return_50d,
    s.vol_20d,
    s.avg_volume_20d,
    s.high_20d,
    s.pct_of_20d_high,
    s.is_volume_spike,
    s.is_breakout_candidate,
    s.momentum_rank
  );